In [ ]:
#import libraries and load data
import pandas as pd
import numpy as np

prices = pd.read_parquet("../data/raw/prices_raw.parquet")
assets = pd.read_csv("../data/clean/dim_asset.csv")

print("Raw prices:", prices.shape)
prices.head()

Raw prices: (8258, 44)


Ticker,000001.SS,^DJI,^FTSE,^GDAXI,^GSPC,^HSI,^IXIC,^N225,BZ=F,CL=F,...,XLE,XLF,XLK,XLP,XLU,XLV,VNQ,XLI,XLY,^VIX
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,11357.509766,NaN,6750.759766,1455.219971,17369.630859,4131.149902,NaN,NaN,NaN,...,6.569212,11.013210,20.577805,11.976416,5.438090,20.273405,NaN,17.982195,11.104394,24.209999
2000-01-04,1406.370972,10997.929688,6665.899902,6586.950195,1399.420044,17072.820312,3901.689941,19002.859375,NaN,NaN,...,6.445557,10.531756,19.533840,11.639161,5.274050,19.813358,NaN,17.485386,10.770344,27.010000
2000-01-05,1409.682007,11122.650391,6535.899902,6502.069824,1402.109985,15846.719727,3877.540039,18542.550781,NaN,NaN,...,6.615585,10.449014,19.243839,11.844802,5.407138,19.635611,NaN,17.407448,10.637876,26.410000
2000-01-06,1463.942017,11253.259766,6447.200195,6474.919922,1403.449951,15153.230469,3727.129883,18168.269531,NaN,NaN,...,6.870621,10.907893,18.605864,12.066888,5.397854,19.698343,NaN,17.641237,10.758829,25.730000
2000-01-07,1516.604004,11522.559570,6504.799805,6780.959961,1441.469971,15405.629883,3882.620117,18193.410156,NaN,NaN,...,6.944043,11.088433,18.930658,12.831865,5.444282,19.938826,NaN,18.313385,11.265663,21.719999


In [2]:
# Rename columns to asset names
name_map = dict(zip(assets["Ticker"], assets["Asset"]))
prices = prices.rename(columns=name_map)

# keep only assets that actually downloaded
assets = assets[assets["Asset"].isin(prices.columns)].reset_index(drop=True)
print(len(assets), "assets in use")

44 assets in use


In [3]:
#Fix non-positive values

log_assets = assets.loc[assets["Transform"] == "log", "Asset"]

for a in log_assets:
    bad = (prices[a] <= 0).sum()
    if bad:
        print(f"{a}: {bad} non-positive values set to NaN")
    prices[a] = prices[a].where(prices[a] > 0)

Crude Oil WTI: 1 non-positive values set to NaN


In [5]:
#Align the calendar to weekdays and forward fill missing values

prices = prices[prices.index.dayofweek < 5]
prices = prices.ffill(limit=3)
print("After weekday filter:", prices.shape)

After weekday filter: (6995, 44)


In [7]:

coverage = pd.DataFrame({
    "First Date": prices.apply(lambda s: s.first_valid_index()),
    "Last Date": prices.apply(lambda s: s.last_valid_index()),
    "Missing %": (prices.isna().mean() * 100).round(1),
})

coverage.index.name = "Asset"          
coverage = coverage.reset_index()      

coverage = coverage.merge(assets[["Asset", "Category"]], on="Asset")
coverage = coverage.sort_values("First Date")
coverage.to_csv("../data/clean/coverage_report.csv", index=False)
coverage

,Asset,First Date,Last Date,Missing %,Category
43,VIX,2000-01-03,2026-10-23,0.0,Volatility
42,Discretionary,2000-01-03,2026-10-23,0.0,Sector
29,5Y Yield,2000-01-03,2026-10-23,0.0,Rates
30,3M Yield,2000-01-03,2026-10-23,0.0,Rates
31,10Y Yield,2000-01-03,2026-10-23,0.0,Rates
34,Energy,2000-01-03,2026-10-23,0.0,Sector
35,Financials,2000-01-03,2026-10-23,0.0,Sector
36,Tech,2000-01-03,2026-10-23,0.0,Sector
17,Dollar Index,2000-01-03,2026-10-23,0.0,Currency
37,Staples,2000-01-03,2026-10-23,0.0,Sector


In [8]:
#Compute returns

def to_returns(p, meta):
    out = {}
    for _, r in meta.iterrows():
        a = r["Asset"]
        if a not in p:
            continue
        if r["Transform"] == "log":
            out[a] = np.log(p[a] / p[a].shift(1))
        else:
            out[a] = p[a].diff()
    return pd.DataFrame(out)

daily = to_returns(prices, assets)

weekly_prices = prices.resample("W-FRI").last()
weekly = to_returns(weekly_prices, assets)

print("Daily:", daily.shape, "| Weekly:", weekly.shape)

Daily: (6995, 44) | Weekly: (1399, 44)


In [9]:
#Remove data errors
price_cols = assets.loc[assets["Transform"] == "log", "Asset"].tolist()
extreme = daily[price_cols].abs() > 0.5
print("Extreme daily moves removed:", int(extreme.sum().sum()))
daily[price_cols] = daily[price_cols].mask(extreme)

extreme_w = weekly[price_cols].abs() > 0.5
weekly[price_cols] = weekly[price_cols].mask(extreme_w)

Extreme daily moves removed: 3


In [10]:
#Save everything

prices.to_parquet("../data/clean/prices_clean.parquet")
weekly_prices.to_parquet("../data/clean/prices_weekly.parquet")
daily.to_parquet("../data/clean/returns_daily.parquet")
weekly.to_parquet("../data/clean/returns_weekly.parquet")
assets.to_csv("../data/clean/dim_asset.csv", index=False)

print("Saved. Daily:", daily.shape, "Weekly:", weekly.shape)

Saved. Daily: (6995, 44) Weekly: (1399, 44)
